In [ ]:
from pathlib import Path
import subprocess
import tempfile
import shutil

# 只改这里
FOLDER = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\trans")

# 翻译设置
SOURCE_LANG = "en"
TARGET_LANG = "zh"

# 默认用 pdf2zh 自带默认翻译服务
# 想指定服务可以写 "google", "deepl", "openai" 等
# SERVICE = None
SERVICE = "google"

# 已经存在 -原文件名.pdf 时是否覆盖
OVERWRITE = False

pdf2zh_cmd = shutil.which("pdf2zh")
if pdf2zh_cmd is None:
    raise RuntimeError("没有找到 pdf2zh。请先运行：pip install pdf2zh")

pdf_files = sorted([
    p for p in FOLDER.glob("*.pdf")
    if p.is_file()
    and not p.name.startswith("-")
    and not p.name.startswith("~$")
])

print("=" * 80)
print(f"待翻译文件夹：{FOLDER}")
print(f"待翻译 PDF 数量：{len(pdf_files)}")
print("=" * 80)

for pdf_path in pdf_files:
    out_path = pdf_path.with_name("-" + pdf_path.name)

    if out_path.exists() and not OVERWRITE:
        print(f"跳过，已存在：{out_path.name}")
        continue

    print("\n" + "-" * 80)
    print(f"开始翻译：{pdf_path.name}")

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        tmp_input = tmpdir / pdf_path.name
        shutil.copy2(pdf_path, tmp_input)

        cmd = [
            pdf2zh_cmd,
            str(tmp_input),
            "-li", SOURCE_LANG,
            "-lo", TARGET_LANG,
        ]

        if SERVICE:
            cmd += ["-s", SERVICE]

        result = subprocess.run(
            cmd,
            cwd=tmpdir,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT
        )

        print(result.stdout[-2000:])

        if result.returncode != 0:
            print(f"失败：{pdf_path.name}")
            continue

        # 优先找双语 dual 文件
        candidates = list(tmpdir.glob(f"{pdf_path.stem}*dual*.pdf"))

        # 兼容不同版本输出名
        if not candidates:
            candidates = list(tmpdir.glob(f"{pdf_path.stem}*zh*.pdf"))
        if not candidates:
            candidates = list(tmpdir.glob(f"{pdf_path.stem}*mono*.pdf"))

        if not candidates:
            print(f"没有找到翻译输出文件：{pdf_path.name}")
            continue

        translated_pdf = candidates[0]

        if out_path.exists() and OVERWRITE:
            out_path.unlink()

        shutil.move(str(translated_pdf), str(out_path))

        print(f"完成：{out_path.name}")

print("\n" + "=" * 80)
print("全部处理完成")
print("=" * 80)

待翻译文件夹：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged
待翻译 PDF 数量：10

--------------------------------------------------------------------------------
开始翻译：_HSC_1.pdf
